# Day 035 — Exercise 5: DigestPipeline

**What you'll build:** The `DigestPipeline` class — `add_source(source) -> DigestPipeline` (fluent builder), `async process() -> dict` (fetch → batch_extract → generate_digest), `run() -> dict` (asyncio.run wrapper for scripts).

**Why it matters:** The assembled capstone class. Every pipeline stage is composed here into a reusable object. `await pipeline.process()` in notebooks; `pipeline.run()` in scripts.

## Provided: All Helper Functions

In [ ]:
import requests
from pathlib import Path

def fetch_text(source: str) -> dict:
    src  = str(source)
    text = None
    kind = 'text'

    if src.startswith('http://') or src.startswith('https://'):
        try:
            response = requests.get(src, timeout=10)
            response.raise_for_status()
            text = response.text
            kind = 'url'
        except Exception as e:
            text = '[fetch error: ' + str(e) + ']'
            kind = 'url_error'
    else:
        try:
            p = Path(src)
            if p.exists() and p.is_file():
                text = p.read_text(encoding='utf-8')
                kind = 'file'
        except Exception:
            pass

    if text is None:
        text = src
        kind = 'text'

    return {
        'source':     src,
        'kind':       kind,
        'content':    text,
        'char_count': len(text),
    }


import json
import ollama
from pydantic import BaseModel, Field

class ArticleInfo(BaseModel):
    title:      str       = Field(description='Topic or title in 3-6 words')
    summary:    str       = Field(description='One sentence summary')
    sentiment:  str       = Field(description='positive, negative, or neutral')
    key_points: list[str] = Field(default_factory=list,
                                  description='Up to 3 key points as short phrases')

def extract_info(doc: dict, model: str = 'llama3.2') -> dict:
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


import asyncio
import json
import ollama

async def async_extract(doc: dict, model: str = 'llama3.2') -> dict:
    client = ollama.AsyncClient()
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = await client.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


async def batch_extract(docs: list, max_concurrent: int = 3,
                        model: str = 'llama3.2') -> list[dict]:
    if not docs:
        return []
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(doc):
        async with sem:
            return await async_extract(doc, model)
    return list(await asyncio.gather(*[_run(d) for d in docs]))


import ollama

def generate_digest(results: list, model: str = 'llama3.2') -> str:
    ok     = [r for r in results if r.get('status') == 'ok']
    errors = [r for r in results if r.get('status') == 'error']
    if not ok:
        return 'No articles extracted successfully (' + str(len(errors)) + ' errors).'
    lines = [
        '=== Auto-Analyst Digest ===',
        str(len(results)) + ' sources processed: '
        + str(len(ok)) + ' ok, ' + str(len(errors)) + ' failed.\n',
    ]
    for i, r in enumerate(ok, 1):
        info      = r.get('info') or {}
        title     = info.get('title',     'Untitled')
        summary   = info.get('summary',   '')
        sentiment = info.get('sentiment', 'unknown')
        kp        = info.get('key_points', [])
        kp_text   = '; '.join(kp[:3]) if kp else ''
        lines.append('[' + str(i) + '] ' + title + '  [' + sentiment + ']')
        lines.append('    ' + summary)
        if kp_text:
            lines.append('    Key points: ' + kp_text)
        lines.append('')
    context  = '\n'.join(lines)
    prompt   = (
        context + '\n\n'
        'Write a 3-4 sentence editorial digest identifying '
        'the main themes, patterns, and key insights across all articles.'
    )
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response['message']['content']

## Your Implementation

In [ ]:
import asyncio

class DigestPipeline:
    """
    End-to-end Auto-Analyst pipeline:
      add_source → process/run → {source_count, ok_count, results, digest}
    """

    def __init__(self, model: str = 'llama3.2', max_concurrent: int = 3):
        # TODO: self.model = model
        # TODO: self.max_concurrent = max_concurrent
        # TODO: self._sources: list = []
        pass

    def add_source(self, source) -> 'DigestPipeline':
        # TODO: self._sources.append(source)
        # TODO: return self
        pass

    async def process(self) -> dict:
        # TODO: docs    = [fetch_text(s) for s in self._sources]
        # TODO: results = await batch_extract(docs, self.max_concurrent, self.model)
        # TODO: digest  = generate_digest(results, self.model)
        # TODO: ok_count = sum(1 for r in results if r.get('status') == 'ok')
        # TODO: return {'source_count': len(docs), 'ok_count': ok_count,
        #               'results': results, 'digest': digest}
        pass

    def run(self) -> dict:
        # TODO: return asyncio.run(self.process())
        pass

## Check Your Work

In [ ]:
import asyncio

async def _run_checks():
    total = 5
    passed = 0

    # Check 1: class defined with correct methods
    try:
        assert 'DigestPipeline' in globals()
        for m in ('add_source', 'process', 'run'):
            assert hasattr(DigestPipeline, m), f'missing method: {m}'
        assert asyncio.iscoroutinefunction(DigestPipeline.process), \
            'process must be async def'
        passed += 1; print('\u2705 Check 1: DigestPipeline with add_source, process, run')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: __init__ stores model, max_concurrent, _sources
    try:
        dp = DigestPipeline(model='llama3.2', max_concurrent=2)
        assert dp.model          == 'llama3.2', f'model: {dp.model!r}'
        assert dp.max_concurrent == 2,          f'max_concurrent: {dp.max_concurrent}'
        assert hasattr(dp, '_sources') and isinstance(dp._sources, list), \
            '_sources should be list'
        passed += 1; print('\u2705 Check 2: __init__ stores model, max_concurrent, _sources')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: add_source returns self and appends
    try:
        dp = DigestPipeline()
        ret = dp.add_source('hello world')
        assert ret is dp, f'add_source should return self, got {type(ret)}'
        assert len(dp._sources) == 1, f'expected 1 source, got {len(dp._sources)}'
        dp.add_source('second doc')
        assert len(dp._sources) == 2, f'expected 2 sources, got {len(dp._sources)}'
        passed += 1; print('\u2705 Check 3: add_source returns self and appends source')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: process() returns correct dict (1 LLM call via batch_extract)
    try:
        dp = DigestPipeline(model='llama3.2', max_concurrent=3)
        dp.add_source('Python is a versatile high-level programming language.')
        output = await dp.process()
        assert isinstance(output, dict), f'expected dict, got {type(output).__name__}'
        for k in ('source_count', 'ok_count', 'results', 'digest'):
            assert k in output, f'missing key: {k}'
        assert output['source_count'] == 1, f'source_count: {output["source_count"]}'
        assert isinstance(output['results'], list) and len(output['results']) == 1
        assert isinstance(output['digest'], str) and output['digest'].strip()
        passed += 1; print(f'\u2705 Check 4: process() returns {{source_count, ok_count, results, digest}}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: fluent chaining
    try:
        dp2 = (
            DigestPipeline()
            .add_source('Artificial intelligence is transforming many industries.')
            .add_source('Climate change requires immediate global action.')
        )
        output2 = await dp2.process()
        assert output2['source_count'] == 2, \
            f'source_count should be 2, got {output2["source_count"]}'
        assert len(output2['results']) == 2
        passed += 1; print('\u2705 Check 5: fluent chaining, 2-source pipeline works')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


await _run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import asyncio

class DigestPipeline:
    def __init__(self, model: str = 'llama3.2', max_concurrent: int = 3):
        self.model          = model
        self.max_concurrent = max_concurrent
        self._sources: list = []

    def add_source(self, source) -> 'DigestPipeline':
        self._sources.append(source)
        return self

    async def process(self) -> dict:
        docs     = [fetch_text(s) for s in self._sources]
        results  = await batch_extract(docs, self.max_concurrent, self.model)
        digest   = generate_digest(results, self.model)
        ok_count = sum(1 for r in results if r.get('status') == 'ok')
        return {
            'source_count': len(docs),
            'ok_count':     ok_count,
            'results':      results,
            'digest':       digest,
        }

    def run(self) -> dict:
        return asyncio.run(self.process())
```

</details>